# Function

In [1]:
import os
import re
import sys
import time
import json
import random
import sqlite3
import numpy as np
import pandas as pd
import multiprocessing as mp
from collections import defaultdict
from func_timeout import func_timeout, FunctionTimedOut
from transformers import AutoTokenizer

In [3]:
def result_callback(result):
    metric_results.append(result)

def run_sqls_parallel(data_name, sql_data, num_cpus=1, iterate_num=1, meta_time_out=30.0):
    with mp.Pool(processes=num_cpus) as pool:
        for i, item in enumerate(sql_data):
            pool.apply_async(
                execute_DB, 
                args=(data_name, item, iterate_num, meta_time_out),
                callback=result_callback
            )
        pool.close()
        pool.join()

def execute_DB(data_name, item, iterate_num, meta_time_out):
    result = {
        'db_id': item["db_id"],
        'question': item["question"],
        'table': item['table'],
        'column': item['column'],
        'foreign_keys': item['foreign_keys'],
        'sql_gt': item['sql_gt'],
        'sql_pred': item['sql_pred'],
        'timeout': False,
        'error': False
    }
    if data_name == "bird":
        result['evidence'] = item['evidence']
    elif data_name == "spider":
        result.update({
            'ori_sql_gt': item['ori_sql_gt'],
            'ori_sql_pred': item['ori_sql_pred']
        })
    
    result_key = 'res' if iterate_num == 1 else 'time_ratio'
    timeout_duration = meta_time_out if iterate_num == 1 else meta_time_out * iterate_num
    
    try:
        sql_result = func_timeout(
            timeout_duration,
            execute_sql,
            args=(data_name, item["sql_pred"], item["sql_gt"], item["db_path"], iterate_num)
        )
        result[result_key] = sql_result['match']
        result['res_pred'] = sql_result['res_pred']
        result['res_gt'] = sql_result['res_gt']
    except FunctionTimedOut:
        result['timeout'] = True
        result[result_key] = 0
    except Exception as e:
        result['error'] = True
        result['error_msg'] = str(e)
        result[result_key] = 0
        
    return result

def clean_abnormal(input_list):
    input_array = np.asarray(input_list)
    mean, std = np.mean(input_array,axis=0), np.std(input_array,axis=0)
    return [x for x in input_array if mean - 3 * std < x < mean + 3 * std]

def execute_time(sql, db_path):
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        start_time = time.time()
        cursor.execute(sql)
        return time.time() - start_time
    
def execute_sql(data_name, sql_pred, sql_gt, db_path, iterate_num):
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        if data_name == "bird":
            cursor.execute(sql_pred)
            res_pred = cursor.fetchall()
            cursor.execute(sql_gt)
            res_gt = cursor.fetchall()
            
            if iterate_num == 1:
                match =  1 if set(res_pred) == set(res_gt) else 0
                return {
                    "match": match,
                    "res_pred": res_pred,
                    "res_gt": res_gt
                }
        
            time_ratio = 0
            if set(res_pred) == set(res_gt):
                diff_list = [
                    execute_time(sql_gt, db_path) / execute_time(sql_pred, db_path)
                    for _ in range(iterate_num)
                ]
                processed_diff_list = clean_abnormal(diff_list)
                time_ratio = sum(processed_diff_list) / len(processed_diff_list)
            return {
                "time_ratio": time_ratio,
                "res_pred": res_pred,
                "res_gt": res_gt
            }

        elif data_name == "spider":
            order_matters = 'order by' in sql_gt.lower()
            for pred in sql_pred:
                pred_passes = 1
                cursor.execute(pred)
                res_pred = cursor.fetchall()
                cursor.execute(sql_gt)
                res_gt = cursor.fetchall()
                
                if not result_eq(res_gt, res_pred, order_matters=order_matters):
                    pred_passes = 0
                if pred_passes == 1:
                    return 1
            return 0

In [4]:
def normalize_sql(sql):
    # merge multiple space to single space
    sql = re.sub(r'\s+', ' ', sql).strip()
    
    # replace "" to ''
    sql = re.sub(r'"([^"]*)"', r"'\1'", sql)

    # Save the content part
    literals = re.findall(r"'[^']*'", sql)
    placeholders = {}
    for i, lit in enumerate(literals):
        key = f"__lit_{i}__"
        placeholders[key] = lit
        sql = sql.replace(lit, key)
    
    # lower the whole sql
    #sql = sql.lower()
    
    # recovore the content part
    for key, lit in placeholders.items():
        sql = sql.replace(key, lit)
    
    # clean the space in  ()
    sql = re.sub(r'\(\s*', '(', sql)
    sql = re.sub(r'\s*\)', ')', sql)
    
    # add ;
    if not sql.endswith(';'):
        sql += ';'
    
    return sql

In [5]:
def deduplicate_foreign_keys(fk_list: list[str]) -> list[str]:
    seen = set()
    result = []

    for fk in fk_list:
        parts = [p.strip() for p in fk.split('=')]
        if len(parts) != 2:
            continue

        key = frozenset(parts)
        if key not in seen:
            seen.add(key)
            result.append(f"{parts[0]} = {parts[1]}")
    
    return result

def get_foreign_keys_relations(schema: dict) -> list:
    foreign_keys = []

    for table_name, columns in schema.items():
        if type(columns) is list:
            columns = {list(col.keys())[0]: list(col.values())[0] for col in columns}
        if type(columns) is str:
                continue
        for column_name, infos in columns.items():
            if column_name == "*":
                continue
            if infos.get("foreign_key", ""):
                foreign_keys.append(f"{table_name}.{column_name} = {infos['foreign_key']}")

    foreign_keys = deduplicate_foreign_keys(foreign_keys)
    
    return foreign_keys

In [6]:
def prepare_sft_dataset(data_name, train_df, top_k, augment_times):
    sql_aug = []
    for train_item in train_df.to_dict(orient='records'):
        if top_k > 0:
            best_schemas_pair = {(k, v) for k, values in train_item['related_schemas'].items() for v in values}
            full_schemas_pair = {(k, v) for k, values in train_item['full_schemas'].items() for v in values}
            remain_schemas_pair = list(full_schemas_pair - best_schemas_pair)
            for _ in range(augment_times):
                sample_count = max(0, top_k - len(best_schemas_pair))
                sample_schemas_pair = set(random.sample(remain_schemas_pair, min(sample_count, len(remain_schemas_pair))))
                
                final_schemas_pair = best_schemas_pair | sample_schemas_pair
                
                final_schemas = defaultdict(dict)
                merge_key = None
                for k, v in final_schemas_pair:
                    if v == "*":
                        merge_key = k
                        continue
                    final_schemas[k][v] = train_item['full_schemas'][k][v]

                if merge_key is not None:
                    final_schemas[merge_key].clear()
                    final_schemas[merge_key] = "*"
                
        
                table = list(final_schemas.keys())
                column = list(final_schemas.values())
        
                sql_item = {
                    "db_id": train_item["db_id"],
                    "question": train_item['question'].strip(),
                    "table": table,
                    "column": column,
                    "sql_gt": normalize_sql(train_item['SQL'].strip()),
                    "foreign_keys": get_foreign_keys_relations(final_schemas)
                }
        
                if data_name == "bird":
                    sql_item['evidence'] = train_item['evidence'].strip()
    
                sql_aug.append(sql_item)
        else:
            best_schemas_pair = {(k, v) for k, values in train_item['related_schemas'].items() for v in values}
            for _ in range(augment_times):
                final_schemas_pair = set(list(best_schemas_pair))
                final_schemas = defaultdict(dict)
                for k, v in final_schemas_pair:
                    final_schemas[k][v] = train_item['full_schemas'][k][v]
        
                table = list(final_schemas.keys())
                column = list(final_schemas.values())
        
                sql_item = {
                    "db_id": train_item["db_id"],
                    "question": train_item['question'].strip(),
                    "table": table,
                    "column": column,
                    "sql_gt": normalize_sql(train_item['SQL'].strip()),
                    "foreign_keys": get_foreign_keys_relations(final_schemas)
                }
        
                if data_name == "bird":
                    sql_item['evidence'] = train_item['evidence'].strip()
                
                sql_aug.append(sql_item)
    return sql_aug

In [7]:
def prepare_dev_dataset(data_name, valid_df, schema_type):
    augmented_valid_data = []

    if data_name == "bird":
        for valid_item in valid_df.to_dict(orient='records'):
            augmented_valid_data.append({
                "question_id": valid_item['question_id'],
                "db_id": valid_item["db_id"],
                "question": valid_item['question'].strip(),
                "table": list(valid_item[schema_type].keys()),
                "column": list(valid_item[schema_type].values()),
                "foreign_keys": get_foreign_keys_relations(valid_item[schema_type]),
                "evidence": valid_item["evidence"].strip(),
                "sql_gt": normalize_sql(valid_item['SQL'].strip()),
                "difficulty": valid_item["difficulty"]
            })
    elif data_name == "spider":
        for idx, valid_item in enumerate(valid_df.to_dict(orient='records')):
            augmented_valid_data.append({
                "question_id": idx,
                "db_id": valid_item["db_id"],
                "question": valid_item['question'].strip(),
                "table": list(valid_item[schema_type].keys()),
                "column": list(valid_item[schema_type].values()),
                "foreign_keys": get_foreign_keys_relations(valid_item[schema_type]),
                "sql_gt": normalize_sql(valid_item['SQL'].strip()),
            })

    return augmented_valid_data

In [8]:
def prepare_test_dataset(data_name, test_df, schema_type):
    augmented_test_data = []
    evaluator = Evaluator()
    
    for idx, test_item in enumerate(test_df.to_dict(orient='records')):
        db_id = test_item["db_id"]
        sql_gt = test_item['SQL'].strip()
        
        db = f"{data_name}/test/test_database/{db_id}/{db_id}.sqlite"
        schema = Schema(get_schema(db))
        tmp_sql = get_sql(schema, sql_gt)
        hardness = evaluator.eval_hardness(tmp_sql)
        
        augmented_test_data.append({
            "question_id": idx,
            "db_id": db_id,
            "question": test_item['question'].strip(),
            "table": list(test_item[schema_type].keys()),
            "column": list(test_item[schema_type].values()),
            "foreign_keys": get_foreign_keys_relations(test_item[schema_type]),
            "sql_gt": normalize_sql(sql_gt.strip()),
            "difficulty": hardness.strip()
        })

    return augmented_test_data

# Bird Dataset

In [9]:
data_name = 'bird'
top_k = 25
augment_times = 1

## Prepare Supervised Training Dataset

In [ ]:
train_df = pd.read_json(f"../../../datasets/{data_name}/train/BIRD-train-more-schema.json", encoding="utf-8")

In [24]:
train_full_schemas = train_df['full_schemas'].tolist()
# analysis the statics of information
comment_list = set()
desc_list = set()
value_string_list = set()
for schema in train_full_schemas:
    for table_key in schema.keys():
        table = schema[table_key]
        for column_key in table.keys():
            column = table[column_key]
            comment_list.add(column['comment'])
            desc_list.add(column['description'])
            value = column['values']
            if len(value) > 0 and type(value[0]) is str:
                value_string_list.update(value)

In [12]:
train_sql_data = prepare_sft_dataset(data_name, train_df, top_k, augment_times)

In [18]:
train_sql_data = set(json.dumps(item, ensure_ascii=True, sort_keys=True) for item in train_sql_data)
train_sql_data = [json.loads(item) for item in train_sql_data]

In [ ]:
with open(f"../../../datasets/{data_name}/train/sft/sql_train.json", "w", encoding="utf-8") as f:
    json.dump(train_sql_data, f, ensure_ascii=True, indent=4)

## Prepare Validation & Testing Dataset

In [ ]:
valid_df = pd.read_json(f"../../../datasets/{data_name}/dev_20240627/retrieved/BIRD-dev-more-schema.json", encoding="utf-8")

In [12]:
schema_type = 'full_schemas'

In [13]:
augmented_valid_data = prepare_dev_dataset(data_name, valid_df, schema_type)

In [ ]:
with open(f"../../../datasets/{data_name}/dev_20240627/retrieved/{schema_type}.json", "w") as f:
    json.dump(augmented_valid_data, f, ensure_ascii=True, indent=4)

## Prepare SL Analysis Dataset

In [ ]:
analysis_df = pd.read_json(f"../../../datasets/{data_name}/dev_20240627/sl_baselines/Codes_subsampled_BIRD_results-more-schema.json", encoding="utf-8")

In [46]:
schema_type = 'selected_columns'

In [47]:
augmented_analysis_data = prepare_dev_dataset(data_name, analysis_df, schema_type)

In [ ]:
with open(f"../../../datasets/{data_name}/dev_20240627/sl_baselines/Codes_{schema_type}.json", "w") as f:
    json.dump(augmented_analysis_data, f, ensure_ascii=True, indent=4)

# Spider

In [ ]:
from itertools import chain
current_path = os.getcwd()
scripts_path = os.path.abspath(os.path.join(current_path, '..', 'scripts'))
sys.path.append(scripts_path)
from utils.spider_tool.evaluation import Evaluator
from utils.spider_tool.process_sql import Schema, get_schema, get_sql

In [10]:
data_name = 'spider'
top_k = 20
augment_times = 1

## Prepare Supervised Training Dataset

In [ ]:
train_df = pd.read_json(f"../../../datasets/{data_name}/train/Spider-train-more-schema.json", encoding="utf-8")

In [14]:
train_sql_data = prepare_sft_dataset(data_name, train_df, top_k, augment_times)

In [20]:
train_sql_data = set(json.dumps(item, ensure_ascii=True, sort_keys=True) for item in train_sql_data)
train_sql_data = [json.loads(item) for item in train_sql_data]

In [ ]:
with open(f"../../../datasets/{data_name}/train/sft/sft_train.json", "w", encoding="utf-8") as f:
    json.dump(train_sql_data, f, ensure_ascii=False, indent=4)

## Prepare Validation Dataset

In [ ]:
valid_df = pd.read_json(f"../../../datasets/{data_name}/dev/retrieved/Spider-dev-more-schema.json", encoding="utf-8")

In [61]:
valid_df.columns

Index(['db_id', 'question', 'SQL', 'SQL_keywords', 'related_schemas',
       'queried_schemas_top10', 'queried_schemas_top15',
       'queried_schemas_top20', 'filtered_columns', 'full_schemas'],
      dtype='object')

In [82]:
schema_type = 'filtered_columns'

In [83]:
augmented_valid_data = prepare_dev_dataset(data_name, valid_df, schema_type)

In [ ]:
with open(f"../../../datasets/{data_name}/dev/retrieved/{schema_type}.json", "w") as f:
    json.dump(augmented_valid_data, f, ensure_ascii=True, indent=4)

## Prepare Test Dataset

In [ ]:
test_df = pd.read_json(f"../../../datasets/{data_name}/test/retrieved/Spider-test-top25-more-schema.json", encoding="utf-8")

In [12]:
test_df.columns

Index(['db_id', 'question', 'SQL', 'related_schemas', 'queried_schemas_top10',
       'queried_schemas_top15', 'queried_schemas_top20',
       'queried_schemas_top25', 'full_schemas'],
      dtype='object')

In [33]:
schema_type = 'queried_schemas_top25'

In [34]:
augmented_test_data = prepare_test_dataset(data_name, test_df, schema_type)

In [ ]:
with open(f"../../../datasets/{data_name}/test/retrieved/{schema_type}.json", "w") as f:
    json.dump(augmented_test_data, f, ensure_ascii=True, indent=4)